# RV, BNS, HAR, And LSTM Plots

This notebook is organized in eight sections for the active coin.

1. Realized volatility overview: RV plot plus a compact RV and sqrt(RV) report.
2. HAR fit comparison: $y_t$ versus $y_t^{HAR}$.
3. HAR residual dynamics: $e_t = y_t^{HAR} - y_t$.
4. HAR and LSTM comparison: $y_t$, $y_t^{HAR}$, and $y_t^{LSTM}$.
5. Residual diagnostics: $e_t$, $e_t^{LSTM}$, squared residuals, and histograms.
6. Error metrics comparison for HAR and LSTM.
7. BNS-based jump diagnostics: jump events over time, jump magnitude, and jump summary metrics.
8. Hawkes intensity plot: rolling daily $\lambda_t$ built from the jump process.

Active coin: `BTC`

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path("..").resolve()
COIN = "BTC"

rv_path = PROJECT_ROOT / "data" / "interim" / "daily_rv" / COIN / "RV_data.parquet"
jump_path = PROJECT_ROOT / "data" / "interim" / "daily_rv_jumps" / COIN / "RV_data.parquet"
jump_summary_path = PROJECT_ROOT / "data" / "interim" / "daily_rv_jumps" / COIN / "data_quality_summary.csv"
hawkes_path = PROJECT_ROOT / "data" / "interim" / "event_5_min" / COIN / "hawkes_feature.parquet"
har_path = PROJECT_ROOT / "results" / "har_forecast" / COIN / "rolling_forecast.parquet"
lstm_path = PROJECT_ROOT / "results" / "lstm_forecast" / COIN / "rolling_forecast.parquet"
config_path = PROJECT_ROOT / "config.yaml"

for required_path, label in [
    (rv_path, "RV file"),
    (jump_path, "BNS jump file"),
    (jump_summary_path, "BNS jump summary file"),
    (hawkes_path, "Hawkes feature file"),
    (har_path, "HAR forecast file"),
    (lstm_path, "LSTM forecast file"),
]:
    if not required_path.exists():
        raise FileNotFoundError(f"{label} not found: {required_path}")

rv_df = pd.read_parquet(rv_path).copy()
jump_df = pd.read_parquet(jump_path).copy()
jump_summary_df = pd.read_csv(jump_summary_path).copy()
hawkes_df = pd.read_parquet(hawkes_path).copy()
har_df = pd.read_parquet(har_path).copy()
lstm_df = pd.read_parquet(lstm_path).copy()

if "date" in rv_df.columns:
    rv_df["date"] = pd.to_datetime(rv_df["date"], errors="coerce")
    rv_df = rv_df.dropna(subset=["date"]).sort_values("date").set_index("date")
elif not isinstance(rv_df.index, pd.DatetimeIndex):
    rv_df.index = pd.to_datetime(rv_df.index, errors="coerce")
    rv_df = rv_df[~rv_df.index.isna()].sort_index()

jump_df["date"] = pd.to_datetime(jump_df["date"], errors="coerce")
jump_df = jump_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

hawkes_df["date"] = pd.to_datetime(hawkes_df["date"], errors="coerce")
hawkes_df = hawkes_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

har_df["date"] = pd.to_datetime(har_df["date"], errors="coerce")
har_df = har_df.dropna(subset=["date"]).sort_values("date")

lstm_df["date"] = pd.to_datetime(lstm_df["date"], errors="coerce")
lstm_df = lstm_df.dropna(subset=["date"]).sort_values("date")

required_har_cols = {"date", "y_true", "y_pred"}
missing_har_cols = required_har_cols - set(har_df.columns)
if missing_har_cols:
    raise ValueError(f"Missing required HAR columns: {sorted(missing_har_cols)}")

required_lstm_cols = {"date", "y_t", "y_t^HAR", "e_t", "e_t^LSTM", "y_t^LSTM"}
missing_lstm_cols = required_lstm_cols - set(lstm_df.columns)
if missing_lstm_cols:
    raise ValueError(f"Missing required LSTM columns: {sorted(missing_lstm_cols)}")

required_hawkes_cols = {"date", "lambda_t"}
missing_hawkes_cols = required_hawkes_cols - set(hawkes_df.columns)
if missing_hawkes_cols:
    raise ValueError(f"Missing required Hawkes columns: {sorted(missing_hawkes_cols)}")

rv_transformation = "log"
if config_path.exists():
    for raw_line in config_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if line.lower().startswith("rv_transformation") and ":" in line:
            rv_transformation = line.split(":", 1)[1].strip().strip('"').strip("'").lower()
            break

if rv_transformation not in {"log", "sqrt"}:
    raise ValueError("RV_transformation in config.yaml must be 'log' or 'sqrt'.")

series_label = "log(RV)" if rv_transformation == "log" else "sqrt(RV)"

C_RV = "#000000"
C_SQRT_RV = "#455A64"
C_JUMP = "#E53935"
C_JUMP_SIZE = "#1E88E5"
C_HAR = "#FB8C00"
C_LSTM = "#1E88E5"
C_RESID = "#E53935"
C_TRUE = "#000000"
C_HAWKES = "#6D4C41"

LAYOUT = dict(template="plotly_white", height=420, xaxis_title="Date")

print(f"Loaded RV rows: {len(rv_df)}")
print(f"Loaded jump rows: {len(jump_df)}")
print(f"Loaded Hawkes rows: {len(hawkes_df)}")
print(f"Loaded HAR forecast rows: {len(har_df)}")
print(f"Loaded LSTM forecast rows: {len(lstm_df)}")
print(f"Forecast scale inferred from config: {series_label}")

Loaded RV rows: 3041
Loaded jump rows: 3024
Loaded Hawkes rows: 2024
Loaded HAR forecast rows: 2008
Loaded LSTM forecast rows: 808
Forecast scale inferred from config: log(RV)


## 1) Realized Volatility Overview

This section shows the daily RV time series and a compact report with both `RV` and `sqrt(RV)` statistics.

In [6]:
rv_plot = rv_df[rv_df["RV"].notna()].copy()
rv_plot["sqrt_RV"] = np.sqrt(rv_plot["RV"].clip(lower=0.0))
rv_plot["log_RV"] = np.log(rv_plot["RV"].clip(lower=1e-12))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rv_plot.index, y=rv_plot["RV"],
    mode="lines", line=dict(color=C_RV, width=1.5), name="RV",
))
fig.update_layout(title=f"{COIN} – RV", yaxis_title="RV", **LAYOUT)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rv_plot.index, y=rv_plot["log_RV"],
    mode="lines", line=dict(color=C_SQRT_RV, width=1.5), name="log(RV)",
))
fig.update_layout(title=f"{COIN} – log(RV)", yaxis_title="log(RV)", **LAYOUT)
fig.show()

rv_summary = pd.DataFrame(
    {
        "series": ["RV", "sqrt(RV)"],
        "mean": [rv_plot["RV"].mean(), rv_plot["sqrt_RV"].mean()],
        "std": [rv_plot["RV"].std(), rv_plot["sqrt_RV"].std()],
        "min": [rv_plot["RV"].min(), rv_plot["sqrt_RV"].min()],
        "max": [rv_plot["RV"].max(), rv_plot["sqrt_RV"].max()],
    }
)

display(rv_summary)

,series,mean,std,min,max
0,RV,0.001677,0.004245,0.000008,0.111207
1,sqrt(RV),0.032899,0.024397,0.002880,0.333477


## 2) $y_t$ vs $y_t^{HAR}$

In [7]:
har_plot = har_df[["date", "y_true", "y_pred"]].copy().sort_values("date")
har_plot["true_series"] = har_plot["y_true"]
har_plot["har_series"]  = har_plot["y_pred"]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=har_plot["date"], y=har_plot["true_series"],
    mode="lines", line=dict(color=C_TRUE, width=1.5), name=series_label,
))
fig.add_trace(go.Scatter(
    x=har_plot["date"], y=har_plot["har_series"],
    mode="lines", line=dict(color=C_HAR, width=1.5), name="HAR prediction",
))
fig.update_layout(
    title=f"{COIN} – {series_label} vs HAR prediction",
    yaxis_title="Value",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    **LAYOUT,
)
fig.show()

## 3) HAR Residuals: $e_t = y_t^{HAR} - y_t$

In [8]:
# Align HAR and LSTM outputs on common dates
compare_df = (
    har_df[["date", "y_true", "y_pred"]]
    .rename(columns={"y_true": "y_t", "y_pred": "y_t^HAR"})
    .merge(
        lstm_df[["date", "y_t", "y_t^HAR", "e_t", "e_t^LSTM", "y_t^LSTM"]],
        on="date",
        suffixes=("_har", "_lstm"),
        how="inner",
    )
    .sort_values("date")
)

# Keep y_t and y_t^HAR from HAR table to preserve the exact HAR benchmark series
compare_df["y_t"] = compare_df["y_t_har"]
compare_df["y_t^HAR"] = compare_df["y_t^HAR_har"]
compare_df["e_t"] = compare_df["y_t^HAR"] - compare_df["y_t"]
compare_df["e_t^LSTM"] = compare_df["y_t^LSTM"] - compare_df["y_t"]

# 5) HAR residual time series
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["e_t"],
    mode="lines", line=dict(color=C_RESID, width=1.5), name="e_t (HAR)",
))
fig.add_hline(y=0.0, line=dict(color="black", dash="dash", width=1.0))
fig.update_layout(
    title=f"{COIN} - HAR residuals (e_t = y_t^HAR - y_t)",
    yaxis_title="Residual",
    **LAYOUT,
)
fig.show()

## 4) $y_t$ vs $y_t^{HAR}$ vs $y_t^{LSTM}$

In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["y_t"],
    mode="lines", line=dict(color=C_TRUE, width=1.5), name="y_t",
))
fig.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["y_t^HAR"],
    mode="lines", line=dict(color=C_HAR, width=1.5), name="y_t^HAR",
))
fig.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["y_t^LSTM"],
    mode="lines", line=dict(color=C_LSTM, width=1.5), name="y_t^LSTM",
))
fig.update_layout(
    title=f"{COIN} - y_t vs y_t^HAR vs y_t^LSTM",
    yaxis_title="Value",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    **LAYOUT,
)
fig.show()

## 5) Residual Diagnostics

In [10]:
MEAN_ROLLING_SIZE = 100

# Graph a) e_t vs e_t^LSTM
fig_a = go.Figure()
fig_a.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["e_t"],
    mode="lines", line=dict(color=C_RESID, width=1.5), name="e_t",
))
fig_a.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["e_t^LSTM"],
    mode="lines", line=dict(color=C_LSTM, width=1.5), name="e_t^LSTM",
))
fig_a.add_hline(y=0.0, line=dict(color="black", dash="dash", width=1.0))
fig_a.update_layout(
    title=f"{COIN} - Graph a) e_t vs e_t^LSTM",
    yaxis_title="Residual",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    **LAYOUT,
)
fig_a.show()

# Squared residuals
compare_df["e_t2"] = compare_df["e_t"] ** 2
compare_df["e_t2^LSTM"] = compare_df["e_t^LSTM"] ** 2

# Smoothed squared residuals
compare_df["e_t2_smooth"] = compare_df["e_t2"].rolling(MEAN_ROLLING_SIZE, min_periods=1).mean()
compare_df["e_t2^LSTM_smooth"] = compare_df["e_t2^LSTM"].rolling(MEAN_ROLLING_SIZE, min_periods=1).mean()

# Graph b) e_t^2 vs e_t^LSTM^2
fig_b = go.Figure()
fig_b.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["e_t2"],
    mode="lines", line=dict(color=C_RESID, width=1.2), name="e_t^2",
))
fig_b.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["e_t2^LSTM"],
    mode="lines", line=dict(color=C_LSTM, width=1.2), name="e_t^LSTM^2",
))
fig_b.update_layout(
    title=f"{COIN} - Graph b) Squared residuals: e_t^2 vs e_t^LSTM^2",
    yaxis_title="Squared residual",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    **LAYOUT,
)
fig_b.show()

# Graph c) Smoothed squared residuals
fig_c = go.Figure()
fig_c.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["e_t2_smooth"],
    mode="lines", line=dict(color=C_RESID, width=2.0),
    name=f"rolling_mean(e_t^2, {MEAN_ROLLING_SIZE})",
))
fig_c.add_trace(go.Scatter(
    x=compare_df["date"], y=compare_df["e_t2^LSTM_smooth"],
    mode="lines", line=dict(color=C_LSTM, width=2.0),
    name=f"rolling_mean(e_t^LSTM^2, {MEAN_ROLLING_SIZE})",
))
fig_c.update_layout(
    title=f"{COIN} - Graph c) Smoothed squared residuals (window={MEAN_ROLLING_SIZE})",
    yaxis_title="Smoothed squared residual",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    **LAYOUT,
)
fig_c.show()

# Graph d) Residual histograms in percent
hist_layout = {k: v for k, v in LAYOUT.items() if k != "xaxis_title"}
fig_d = go.Figure()
fig_d.add_trace(go.Histogram(
    x=compare_df["e_t"],
    histnorm="percent",
    marker_color=C_RESID,
    opacity=0.60,
    name="e_t",
))
fig_d.add_trace(go.Histogram(
    x=compare_df["e_t^LSTM"],
    histnorm="percent",
    marker_color=C_LSTM,
    opacity=0.60,
    name="e_t^LSTM",
))
fig_d.update_layout(
    title=f"{COIN} - Graph d) Residual histograms: e_t and e_t^LSTM",
    xaxis_title="Residual",
    yaxis_title="Frequency (%)",
    bargap=0.05,
    barmode="overlay",
    **hist_layout,
)
fig_d.show()

## 6) Error Metrics Comparison (HAR vs LSTM)

In [11]:
def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return np.mean((y_pred - y_true) ** 2)


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return np.mean(np.abs(y_pred - y_true))


def hmse(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.clip(np.abs(y_true), eps, None)
    return np.mean(((y_pred - y_true) / denom) ** 2)


def hmae(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.clip(np.abs(y_true), eps, None)
    return np.mean(np.abs(y_pred - y_true) / denom)


def qlike_from_log_rv(y_true, y_pred, eps=1e-12):
    rv_true = np.exp(np.asarray(y_true, dtype=np.float64))
    rv_pred = np.exp(np.asarray(y_pred, dtype=np.float64))
    rv_true = np.clip(rv_true, eps, None)
    rv_pred = np.clip(rv_pred, eps, None)
    ratio = rv_true / rv_pred
    return np.mean(ratio - np.log(ratio) - 1.0)


y_true = compare_df["y_t"].to_numpy(dtype=np.float64)
y_har = compare_df["y_t^HAR"].to_numpy(dtype=np.float64)
y_lstm = compare_df["y_t^LSTM"].to_numpy(dtype=np.float64)

metrics_names = ["mse", "mae", "hmse", "hmae", "qlike"]
metrics_har = [
    mse(y_true, y_har),
    mae(y_true, y_har),
    hmse(y_true, y_har),
    hmae(y_true, y_har),
    qlike_from_log_rv(y_true, y_har),
]
metrics_lstm = [
    mse(y_true, y_lstm),
    mae(y_true, y_lstm),
    hmse(y_true, y_lstm),
    hmae(y_true, y_lstm),
    qlike_from_log_rv(y_true, y_lstm),
]

metrics_table = pd.DataFrame(
    {"metric": metrics_names, "HAR": metrics_har, "LSTM": metrics_lstm}
)
display(metrics_table)

fig = go.Figure()
fig.add_trace(go.Bar(x=metrics_names, y=metrics_har, name="HAR", marker_color=C_HAR))
fig.add_trace(go.Bar(x=metrics_names, y=metrics_lstm, name="LSTM", marker_color=C_LSTM))
fig.update_layout(
    title=f"{COIN} - Error metrics comparison (HAR vs LSTM)",
    xaxis_title="Metric",
    yaxis_title="Value",
    barmode="group",
    template="plotly_white",
    height=460,
)
fig.show()

,metric,HAR,LSTM
0,mse,0.551969,1.096451
1,mae,0.543447,0.770098
2,hmse,0.010623,0.018847
3,hmae,0.071989,0.100114
4,qlike,0.353582,0.694157


## 7) BNS Jump Diagnostics

This section reports jump-event metrics and shows the detected jumps over time, followed by jump magnitudes over time.

In [12]:
jump_metrics = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Number of jumps",
            "Alpha used",
            "Jump ratio",
        ],
        "value": [
            int(len(jump_df)),
            int(jump_df["jump_indicator"].sum()),
            float(jump_df["alpha"].iloc[0]),
            float(jump_df["jump_indicator"].mean()),
        ],
    }
)

display(jump_metrics)
display(jump_summary_df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=jump_df["date"],
    y=jump_df["jump_indicator"],
    mode="lines",
    line=dict(color=C_JUMP, width=1.5),
    name="Jump indicator",
))
fig.update_layout(
    title=f"{COIN} – BNS jump events over time",
    yaxis_title="Jump indicator",
    **LAYOUT,
)
fig.update_yaxes(tickmode="array", tickvals=[0, 1])
fig.show()

,metric,value
0,Total rows,3024.000000
1,Number of jumps,608.000000
2,Alpha used,0.010000
3,Jump ratio,0.201058


,asset,expected_obs_per_day,coverage_threshold,jump_test_alpha,init_date,end_date,valid_start,valid_end,total_days_in_valid_interval,days_with_missing_data,pct_days_with_missing_data,days_completely_missed,pct_days_completely_missed,n_missing_dates_between,missing_dates_between,rows_saved,jump_days_detected
0,BTC,288,0.9,0.01,2017-08-18 00:00:00+00:00,2025-12-31 00:00:00+00:00,2017-08-17 00:00:00+00:00,2025-12-31 00:00:00+00:00,3059,35,1.144165,0,0.0,35,2017-08-17;2017-09-06;2017-12-04;2017-12-18;20...,3024,608


### Jump Magnitude Over Time

Jump magnitude is reported with `JV`, with detected jump dates highlighted explicitly.

In [13]:
jump_events = jump_df.loc[jump_df["jump_indicator"] == 1].copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=jump_df["date"], y=jump_df["JV"],
    mode="lines", line=dict(color=C_SQRT_RV, width=1.2), name="JV",
))
fig.add_trace(go.Scatter(
    x=jump_events["date"], y=jump_events["JV"],
    mode="markers",
    marker=dict(color=C_JUMP_SIZE, size=7),
    name="Detected jumps",
))
fig.update_layout(
    title=f"{COIN} – Jump magnitude over time",
    yaxis_title="JV",
    **LAYOUT,
)
fig.show()

## 8) Hawkes Intensity Plot

This section shows the rolling Hawkes daily intensity feature `lambda_t` computed from the jump process.

In [14]:
hawkes_plot = hawkes_df[["date", "lambda_t"]].copy().sort_values("date")

hawkes_summary = pd.DataFrame(
    {
        "metric": ["Rows", "Mean lambda_t", "Std lambda_t", "Min lambda_t", "Max lambda_t"],
        "value": [
            int(len(hawkes_plot)),
            float(hawkes_plot["lambda_t"].mean()),
            float(hawkes_plot["lambda_t"].std()),
            float(hawkes_plot["lambda_t"].min()),
            float(hawkes_plot["lambda_t"].max()),
        ],
    }
)

display(hawkes_summary)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hawkes_plot["date"],
    y=hawkes_plot["lambda_t"],
    mode="lines",
    line=dict(color=C_HAWKES, width=1.5),
    name="lambda_t",
))
fig.update_layout(
    title=f"{COIN} – Hawkes intensity over time",
    yaxis_title="lambda_t",
    **LAYOUT,
)
fig.show()

,metric,value
0,Rows,2024.000000
1,Mean lambda_t,0.832527
2,Std lambda_t,0.180717
3,Min lambda_t,0.455018
4,Max lambda_t,1.450707
